In [1]:
import sys
import os

# Add paths to import from long_form_factuality
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
lff_root = os.path.join(project_root, "long_form_factuality")
for path in [project_root, lff_root]:
    if path not in sys.path:
        sys.path.insert(0, path)


from vllm_wrapper import VLLMRaterModel
from eval.safe.rate_atomic_fact import check_atomic_fact

In [2]:
import json
atomic_fact_data = []
# In Jupyter notebooks, use getcwd() instead of __file__
path = os.path.join(os.getcwd(), "data_for_git", "atomic_facts.jsonl")
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        atomic_fact_data.append(json.loads(line))

In [3]:
total_facts = len([fact for query in atomic_fact_data for sentence_facts in query["results"]["all_atomic_facts"] for fact in sentence_facts["atomic_facts"]])
print(total_facts)
print(len(atomic_fact_data))
atomic_fact_data[0]["results"]["all_atomic_facts"][0]["atomic_facts"]

In [ ]:
llm = VLLMRaterModel()
import threading
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import json

MAX_WORKERS = 1000


# Thread-safe list to collect results
results = []
results_lock = threading.Lock()
error_log = []
error_log_lock = threading.Lock()
pbar_lock = threading.Lock()

def worker(full_dict, pbar):
    try:
        # Extract prompt logic...
        bio_person = full_dict["prompt"].split("Tell me a bio of ")[1]
        
        all_atomic_facts = full_dict["results"]["all_atomic_facts"]
        for sentence_facts in all_atomic_facts:
            for fact_index, fact in enumerate(sentence_facts["atomic_facts"]):
                # 'llm' is captured from outer scope or passed via partial
                rating = check_atomic_fact(fact, bio_person, llm, max_steps=2)[0].answer
                sentence_facts["atomic_facts"][fact_index] = {"fact": fact, "rating": rating}
                with pbar_lock:
                    pbar.update(1)
        
        return ("success", full_dict)
    except Exception as e:
        return ("error", f"Error processing response {full_dict.get('id', '?')}: {e}")

first_n = None

total_facts = len([fact for query in atomic_fact_data for sentence_facts in query["results"]["all_atomic_facts"] for fact in sentence_facts["atomic_facts"]])

tasks = atomic_fact_data[:first_n] if first_n else atomic_fact_data

# Define output file for incremental saving
output_file = os.path.join(os.getcwd(), "data_for_git", "rated_facts.jsonl")

# 2. Run with Executor
# We create the pbar outside, then pass it into every worker
with tqdm(total=total_facts, desc="Rating Facts") as pbar:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # Submit all tasks
        futures = [executor.submit(worker, task, pbar) for task in tasks]
        
        # Process results as they complete
        for future in as_completed(futures):
            status, payload = future.result()
            
            if status == "success":
                # Save to disk immediately (append mode)
                with open(output_file, "a", encoding="utf-8") as f:
                    f.write(json.dumps(payload) + "\n")
                
                with results_lock:
                    results.append(payload)
            else:
                with error_log_lock:
                    error_log.append(payload)

# 3. Save Logs
with open(os.getcwd() + "/data_for_git/fact_rating_log.txt", "w") as f:
    for error in error_log:
        f.write(error + "\n")

print(f"Processed {len(results)} responses successfully.")

In [23]:
#write results to file with utf8 encoding
with open(os.getcwd() + "/data_for_git/rated_facts.jsonl", "w", encoding="utf-8") as f:
    for result in results:
        f.write(json.dumps(result) + "\n")
